# 04 Backtest: incumbent vs ML policy on the 8 held-out weeks, and the value model

1. **The world.** Re-run the generator's own simulation (unmodified, same seed) to rebuild the
   exact hourly true demand of the holdout and the exact store state on the evening before it
   (`bigbox_ml/backtest.py`). Check that it is the same world the pipeline ingested.
2. **Replay** the same demand hour by hour under (a) the incumbent rule and (b) the ML policy:
   the `@champion` forecast for each of the next 14 days + variability-based safety stock
   (`policy.yaml`), at three service levels. Supply (short / late deliveries) and damage draws are
   seeded by day, so both policies face the same suppliers.
3. **Compare** lost sales, OSA, in-stock, shrink and days of supply (slow movers separately), and
   scale the deltas with `src/value/assumptions.yaml`.

What this does *not* credit: intraday visibility (the simulator has no backroom stock and no
expedited delivery, so seeing a stockout sooner changes nothing here). Policy (b) also reads the
history columns of the feature table, which were observed under the incumbent's stockouts, so
its inputs are as censored as history was; a closed loop would only improve them.

In [ ]:
import datetime as dt
import json
import os
import sys
import time

import mlflow
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

sys.path[:0] = [os.path.abspath("../generator"), os.path.abspath("../value")]
import value_model  # noqa: E402
from bigbox_gen.config import config_sha256, load_config  # noqa: E402
from bigbox_ml import backtest as bt  # noqa: E402
from bigbox_ml import policy as pol  # noqa: E402
from bigbox_ml.features import FEATURES, horizon_frame, planned_promotions  # noqa: E402

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.text("run_id", "")
CAT = dbutils.widgets.get("catalog")
MODEL = f"{CAT}.ml.demand_lgbm"
FT = f"{CAT}.ml.store_sku_features"
SEED = 20261023
RUN_ID = dbutils.widgets.get("run_id") or None  # the job passes {{job.run_id}}

cfg = load_config("../generator/config.yaml")
manifest = json.loads(
    dbutils.fs.head(f"/Volumes/{CAT}/raw/landing/_manifest/generator_run.json", 100_000)
)
assert manifest["config_sha256"] == config_sha256(cfg), (
    "config differs from the run that wrote the landing data"
)
print(
    f"generator config sha256 {config_sha256(cfg)[:16]}... = the landing manifest's "
    f"(seed {manifest['seed']})"
)
cal, m, _ = bt.world(cfg)
FIRST = cfg["calendar"]["history_end"] - dt.timedelta(days=55)
LAST = cfg["calendar"]["history_end"]
first, last = cal.index(FIRST), cal.index(LAST)
P = pol.Policy.load("policy.yaml")
client = MlflowClient(registry_uri="databricks-uc")
VERSION = client.get_model_version_by_alias(MODEL, "champion").version
print(f"replay {FIRST} .. {LAST} ({last - first + 1} days), champion = version {VERSION}")

generator config sha256 4968c78f547134ce... = the landing manifest's (seed 20261023)
replay 2026-08-06 .. 2026-09-30 (56 days), champion = version 1


## Segments: slow movers by observed sales over the 8 weeks before the holdout

In [ ]:
cell_of = {
    (m.store_id[st], m.sku_id[sk]): c
    for c, (st, sk) in enumerate(zip(m.cell_store, m.cell_sku, strict=True))
}
velocity = (
    spark.read.table(f"{CAT}.gold.store_sku_daily")
    .where(
        F.col("date").between(
            F.lit(FIRST - dt.timedelta(days=56)), F.lit(FIRST - dt.timedelta(days=1))
        )
    )
    .groupBy("store_id", "sku_id")
    .agg(F.avg("units_sold").alias("v"))
    .toPandas()
)
slow = np.zeros(m.n_cells, dtype=bool)
for s, k, v in velocity.itertuples(index=False):
    slow[cell_of[(s, k)]] = v < P.slow_mover_units_per_day
group = slow.astype(np.int64) * 2 + m.sku_perishable[m.cell_sku].astype(np.int64)
GROUPS = [("other", False), ("other", True), ("slow", False), ("slow", True)]
cells = bt.Cells.of(cfg, m, group, 4)
print(
    f"slow movers (< {P.slow_mover_units_per_day} units/day): {slow.sum():,} of {m.n_cells:,} cells"
)

slow movers (< 1.0 units/day): 8,321 of 28,005 cells


## 1. The world: re-run the simulation and capture the holdout

In [ ]:
t0 = time.time()
cap = bt.capture(cfg, cells, first, last)
print(
    f"simulated to {LAST} in {time.time() - t0:.0f} s; "
    f"start state: {int(cap.start.inv.sum()):,} units "
    f"on hand, {int(cap.start.on_order.sum()):,} on order"
)


PERISHABLE_GROUPS = [g for g, (_, p) in enumerate(GROUPS) if p]
SLOW_GROUPS = [g for g, (seg, _) in enumerate(GROUPS) if seg == "slow"]


def kpis(days: list[np.ndarray], groups=None) -> dict:
    """KPIs over the replayed days; groups = metric groups to include (default all)."""
    arr = np.stack(days)
    arr = arr if groups is None else arr[:, groups, :]
    t = dict(zip(bt.METRICS, arr.sum(axis=(0, 1)), strict=True))
    pm = dict(
        zip(bt.METRICS, np.stack(days)[:, PERISHABLE_GROUPS, :].sum(axis=(0, 1)), strict=True)
    )
    return {
        "in_stock_pct": t["in_stock_at_close"] / t["cell_days"],
        "osa": 1 - t["stockout_hours"] / t["trading_hours"],
        "lost_units_pct": t["lost_units"] / t["demand_units"],
        "lost_sales": t["lost_sales"],
        "units_sold": t["sold_units"],
        "shrink_pct": (pm["expired_units"] + pm["damaged_units"]) / pm["received_units"],
        "waste_cost": t["waste_cost"],
        "days_of_supply": t["closing_units"] / t["sold_units"],
        "avg_inventory_cost": t["closing_cost"] / len(days),
        "order_lines": t["order_lines"],
    }


gold = (
    spark.read.table(f"{CAT}.gold.store_sku_daily")
    .where(F.col("date").between(F.lit(FIRST), F.lit(LAST)))
    .agg(
        (F.sum(F.col("in_stock_at_close").cast("int")) / F.count("*")).alias("in_stock_pct"),
        (1 - F.sum("stockout_hours") / F.sum("trading_hours")).alias("osa"),
        F.sum("units_sold").alias("units_sold"),
    )
    .first()
)
hist = kpis(cap.history)
print("same world? the generator's history vs gold (what the pipeline ingested), holdout weeks:")
for k in ("in_stock_pct", "osa", "units_sold"):
    print(f"  {k:<14} generator {hist[k]:,.6f}   gold {gold[k]:,.6f}")
print("(units: gold excludes the rows silver quarantined, see evidence/01_lakeflow.md)")

captured days 2026-08-06 .. 2026-09-30: 28005 cells
simulated to 2026-09-30 in 58 s; start state: 492,718 units on hand, 507,161 on order
same world? the generator's history vs gold (what the pipeline ingested), holdout weeks:
  in_stock_pct   generator 0.909060   gold 0.909060
  osa            generator 0.926274   gold 0.926274
  units_sold     generator 6,005,368.000000   gold 5,965,472.000000
(units: gold excludes the rows silver quarantined, see evidence/01_lakeflow.md)


## 2. ML forecasts for every night of the holdout (the `@champion`, 14 days ahead)

In [ ]:
t0 = time.time()
K = P.horizon_days
model = mlflow.pyfunc.load_model(f"models:/{MODEL}/{VERSION}")
origins = (
    spark.read.table(FT)
    .where(
        F.col("date").between(
            F.lit(FIRST + dt.timedelta(days=1)), F.lit(LAST + dt.timedelta(days=1))
        )
    )
    .select("store_id", "sku_id", "date", *FEATURES)
    .toPandas()
)
origins["date"] = pd.to_datetime(origins["date"]).dt.date
promos = planned_promotions(
    spark.read.table(f"{CAT}.silver.promotions"),
    spark.read.table(f"{CAT}.silver.stores"),
    str(FIRST),
    str(LAST + dt.timedelta(days=K + 1)),
).toPandas()
n_days = last - first + 1
forecasts = np.zeros((n_days, m.n_cells, K), dtype=np.float32)
sigma = np.zeros((n_days, m.n_cells))
for i in range(n_days):
    o = origins[origins["date"] == FIRST + dt.timedelta(days=i + 1)].reset_index(drop=True)
    idx = np.array([cell_of[(s, k)] for s, k in zip(o["store_id"], o["sku_id"], strict=True)])
    f = np.asarray(model.predict(horizon_frame(o, promos, K)[FEATURES]), dtype=float).reshape(
        len(o), K
    )
    forecasts[i, idx] = f
    sigma[i, idx] = pol.daily_sigma(f[:, 0], o["std_28"].to_numpy())
print(f"{n_days} nights x {m.n_cells:,} cells x {K} days forecast in {time.time() - t0:.0f} s")

{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '113e234f6e4849b99c082963dae58fa3'}}

Unexpected status resolving file endpoints (403, request ID ba4ba7ec-f0b4-4ea9-b741-45c3c96e9a88), using the default endpoints.


56 nights x 28,005 cells x 14 days forecast in 223 s


## 3. Replay

In [ ]:
runs = {}
t0 = time.time()
runs[("incumbent", None)] = bt.replay(
    cfg, cells, cap, bt.Incumbent(cfg, cells, cap.start.sales_ring), SEED
)
for level in P.service_level_sensitivity:
    rule = bt.ForecastPolicy(cells, forecasts, sigma, P.with_service_level(level))
    runs[("ml", level)] = bt.replay(cfg, cells, cap, rule, SEED)
print(f"{len(runs)} replays in {time.time() - t0:.0f} s")

rows = []
for (policy, level), days in [(("history", None), cap.history), *runs.items()]:
    for i, day in enumerate(days):
        for g, (segment, perishable) in enumerate(GROUPS):
            rows.append(
                {
                    "policy": policy,
                    "service_level": level,
                    "date": FIRST + dt.timedelta(days=i),
                    "segment": segment,
                    "perishable": perishable,
                    **dict(zip(bt.METRICS, map(float, day[g]), strict=True)),
                    "run_id": str(RUN_ID),
                }
            )
spark.createDataFrame(pd.DataFrame(rows)).createOrReplaceTempView("bt")
cols = ", ".join(c.name for c in spark.read.table(f"{CAT}.ml.backtest_daily").schema)
spark.sql(f"INSERT OVERWRITE {CAT}.ml.backtest_daily SELECT {cols} FROM bt")
print(f"ml.backtest_daily: {len(rows):,} rows")

4 replays in 25 s
ml.backtest_daily: 1,120 rows


## Results

In [ ]:
table = []
for (policy, level), days in [(("history", None), cap.history), *runs.items()]:
    k = kpis(days)
    s = kpis(days, SLOW_GROUPS)
    table.append(
        {
            "policy": policy,
            "service_level": level,
            **k,
            "slow_days_of_supply": s["days_of_supply"],
            "slow_osa": s["osa"],
        }
    )
res = pd.DataFrame(table)
pd.set_option("display.width", 250)
print(res.round(4).to_string(index=False))

base = res[res["policy"] == "incumbent"].iloc[0]
new = res[(res["policy"] == "ml") & (res["service_level"] == P.service_level)].iloc[0]
print(f"\nML policy at service level {P.service_level} vs incumbent (same demand, same suppliers):")
print(
    f"  lost sales          {base.lost_sales:,.0f} -> {new.lost_sales:,.0f} "
    f"({new.lost_sales / base.lost_sales - 1:+.1%})"
)
print(
    f"  on-shelf avail.     {base.osa:.4f} -> {new.osa:.4f} ({(new.osa - base.osa) * 100:+.2f} pts)"
)
print(
    f"  in-stock at close   {base.in_stock_pct:.4f} -> {new.in_stock_pct:.4f} "
    f"({(new.in_stock_pct - base.in_stock_pct) * 100:+.2f} pts)"
)
print(
    f"  perishable shrink   {base.shrink_pct:.4f} -> {new.shrink_pct:.4f} "
    f"({new.shrink_pct / base.shrink_pct - 1:+.1%})"
)
print(f"  days of supply      {base.days_of_supply:.2f} -> {new.days_of_supply:.2f}")
print(
    f"  slow movers DoS     {base.slow_days_of_supply:.2f} -> {new.slow_days_of_supply:.2f} "
    f"(OSA {base.slow_osa:.4f} -> {new.slow_osa:.4f})"
)

   policy  service_level  in_stock_pct    osa  lost_units_pct  lost_sales  units_sold  shrink_pct  waste_cost  days_of_supply  avg_inventory_cost  order_lines  slow_days_of_supply  slow_osa
  history            NaN        0.9091 0.9263          0.0376  5429948.08   6005368.0      0.1058  1035706.32          4.5226        2357814.0061     414383.0              12.8738    0.8395
incumbent            NaN        0.9094 0.9266          0.0375  5416201.08   6005968.0      0.1056  1034935.22          4.5183        2354585.7577     414753.0              12.8672    0.8408
       ml           0.90        0.9132 0.9366          0.0519  4354528.22   5916582.0      0.0656   634426.33          3.5137        2221603.2891     393380.0              12.7287    0.8974
       ml           0.95        0.9292 0.9482          0.0407  3588350.77   5986337.0      0.0698   698818.82          3.8049        2463650.3484     395335.0              13.0214    0.9131
       ml           0.98        0.9434 0.9582     

## Value model (`src/value/assumptions.yaml`)

In [ ]:
A = value_model.load("../value/assumptions.yaml")
revenue = (
    spark.read.table(f"{CAT}.gold.store_sku_daily")
    .where(F.col("date").between(F.lit(cfg["calendar"]["history_start"]), F.lit(LAST)))
    .agg(F.sum("revenue"))
    .first()[0]
)


def totals(days):
    t = dict(zip(bt.METRICS, np.stack(days).sum(axis=(0, 1)), strict=True))
    return value_model.PolicyTotals(
        len(days), t["lost_sales"], t["lost_margin"], t["waste_cost"], t["closing_cost"]
    )


b, n = totals(runs[("incumbent", None)]), totals(runs[("ml", P.service_level)])
v = value_model.annual_value(b, n, revenue, A)
print(
    f"modelled assortment revenue, 365 days of history (gold): ${revenue:,.0f}; scale to "
    f"${A['company_annual_revenue_usd']:,.0f} = x{v['scale_to_company']:.2f}; "
    f"realisation {v['realisation']:.0%}"
)
for k in (
    "lost_sales_recovered",
    "margin_recovered",
    "waste_avoided",
    "carrying_cost_avoided",
    "total",
):
    print(f"  {k:<24} ${v[k]:>14,.0f} / year")
sens = pd.DataFrame(value_model.sensitivity(b, n, revenue, A)).pivot(
    index="realisation", columns="company_annual_revenue_usd", values="total"
)
print("\nannual value ($) by realisation x company revenue:")
print(sens.map(lambda x: f"{x:,.0f}").to_string())

modelled assortment revenue, 365 days of history (gold): $280,727,195; scale to $1,000,000,000 = x3.56; realisation 50%
  lost_sales_recovered     $    21,219,297 / year
  margin_recovered         $     7,125,122 / year
  waste_avoided            $     3,901,935 / year
  carrying_cost_avoided    $       -38,851 / year
  total                    $    10,988,206 / year

annual value ($) by realisation x company revenue:
company_annual_revenue_usd 8.000000e+08 1.000000e+09 1.200000e+09
realisation                                                      
0.25                          4,395,283    5,494,103    6,592,924
0.50                          8,790,565   10,988,206   13,185,848
0.75                         13,185,848   16,482,309   19,778,771
1.00                         17,581,130   21,976,413   26,371,695


In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            "replay": [str(FIRST), str(LAST)],
            "champion_version": str(VERSION),
            "history_vs_gold": {
                k: [round(float(hist[k]), 6), round(float(gold[k]), 6)]
                for k in ("in_stock_pct", "osa", "units_sold")
            },
            "results": json.loads(res.to_json(orient="records")),
            "value": {k: round(float(x), 2) for k, x in v.items()},
            "modelled_revenue": round(float(revenue), 2),
        }
    )
)

{"replay": ["2026-08-06", "2026-09-30"], "champion_version": "1", "history_vs_gold": {"in_stock_pct": [0.90906, 0.90906], "osa": [0.926274, 0.926274], "units_sold": [6005368.0, 5965472.0]}, "results": [{"policy": "history", "service_level": null, "in_stock_pct": 0.9090596067, "osa": 0.9262735973, "lost_units_pct": 0.0376413803, "lost_sales": 5429948.080000009, "units_sold": 6005368.0, "shrink_pct": 0.1057755914, "waste_cost": 1035706.3200000016, "days_of_supply": 4.5225864593, "avg_inventory_cost": 2357814.0060714297, "order_lines": 414383.0, "slow_days_of_supply": 12.8737721978, "slow_osa": 0.8395312818}, {"policy": "incumbent", "service_level": null, "in_stock_pct": 0.9093873543, "osa": 0.9265818968, "lost_units_pct": 0.0375452305, "lost_sales": 5416201.080000011, "units_sold": 6005968.0, "shrink_pct": 0.1056117009, "waste_cost": 1034935.2200000018, "days_of_supply": 4.5182668306, "avg_inventory_cost": 2354585.757678571, "order_lines": 414753.0, "slow_days_of_supply": 12.8671731864, 